# Tucker / HOSVD 基礎実装

このNotebookでは、**TensorLyを使う前に、PyTorchだけでHOSVD/Tucker分解の基本処理を自作する**。

目的は、mode-\(n\) unfolding / folding、mode-\(n\) product、factor matrix、core tensor、再構成の関係を自分で実装して確認すること。

`hosvd(X, ranks)` は `ranks` のkeyで分解対象modeを指定できる汎用形にする。全modeを指定すれば通常のtruncated HOSVD、modeの一部だけを指定すればpartial HOSVDとして扱う。

自作後は、完全rankでの再構成、低rankでの再構成誤差、factor matrixの直交性を確認し、TensorLyの対応APIとも照合する。

> TensorLyをPyTorch Tensorのまま使う場合は `tensorly.set_backend("pytorch")` を指定できる。


## 1. 分解対象テンソル

この3階テンソル `X` をTucker/HOSVDの自作実装に使用する。


In [1]:
import torch

# このNotebookで自作した処理は src へ共通化済み。
#   unfold / fold / mode_dot            -> nn_compression.tensor
#   hosvd / reconstruct_tucker          -> nn_compression.compression.tucker
#   relative_frobenius_error            -> nn_compression.metrics
from nn_compression.compression import (
    hosvd,
    reconstruct_tucker,
    truncated_svd,
)
from nn_compression.metrics import relative_frobenius_error
from nn_compression.tensor import fold, mode_dot, unfold

X = torch.tensor(
    [
        [[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0]],
        [[2.0, 1.0], [4.0, 3.0], [6.0, 5.0], [8.0, 7.0]],
        [[1.0, 3.0], [2.0, 4.0], [3.0, 5.0], [4.0, 6.0]],
    ],
    dtype=torch.float32,
)

print(X)
print("shape:", X.shape)


tensor([[[1., 2.],
         [3., 4.],
         [5., 6.],
         [7., 8.]],

        [[2., 1.],
         [4., 3.],
         [6., 5.],
         [8., 7.]],

        [[1., 3.],
         [2., 4.],
         [3., 5.],
         [4., 6.]]])
shape: torch.Size([3, 4, 2])


## 2. mode-\(n\) unfolding


In [2]:
# 実装は nn_compression/tensor/operations.py の unfold
for mode in range(X.ndim):
    print(f"mode={mode}: {tuple(unfold(X, mode).shape)}")

### ライブラリでは

TensorLyには同じ目的の `tensorly.base.unfold` が用意されている。

```python
import tensorly as tl
tl.set_backend("pytorch")

X_mode0 = tl.unfold(X, mode=0)
```

PyTorchには **mode-\(n\) unfolding専用の同等APIはない**。`torch.Tensor.unfold` はスライディングウィンドウを取り出す別の処理なので、ここでいうunfoldingとは異なる。


## 3. folding


In [3]:
# 実装は nn_compression/tensor/operations.py の fold
for mode in range(X.ndim):
    restored = fold(unfold(X, mode), mode, X.shape)
    print(f"mode={mode}: restored == X -> {torch.equal(restored, X)}")

### 確認

各modeで次が成立することを確認する。

\[
\mathrm{fold}(\mathrm{unfold}(X,n),n,\mathrm{shape}(X)) = X
\]

### ライブラリでは

TensorLyには `tensorly.base.fold` が用意されている。

```python
X_mode0 = tl.unfold(X, mode=0)
X_restored = tl.fold(X_mode0, mode=0, shape=X.shape)
```

PyTorchにはmode-\(n\) folding専用の同等APIはない。


## 4. mode-\(n\) product


In [ ]:
# 実装は nn_compression/tensor/operations.py の mode_dot
# mode=0 なので matrix.shape == (新しいmodeのサイズ, X.shape[0]) == (?, 3)
matrix = torch.tensor([[1.0, 2.0, 1.0]], dtype=torch.float32)
Y = mode_dot(X, matrix, mode=0)
print("matrix.shape:", tuple(matrix.shape))
print("Y.shape:", tuple(Y.shape))
print(Y)

### ライブラリでは

TensorLyには `tensorly.tenalg.mode_dot` が用意されている。

```python
from tensorly.tenalg import mode_dot

Y = mode_dot(X, matrix, mode=0)
```

PyTorchにはn-mode product専用の同等APIはない。一般的なテンソル縮約には `torch.tensordot` や `torch.einsum` がある。


## 5. truncated HOSVD / partial HOSVD


In [ ]:
# 実装は nn_compression/compression/tucker.py の hosvd
# factor は逐次更新した core ではなく、常に元の X の unfolding から求める
core_full, factors_full = hosvd(X, {0: 2, 1: 2, 2: 2})
core_partial, factors_partial = hosvd(X, {0: 2, 1: 2})

print("full   core:", tuple(core_full.shape), "factors:", sorted(factors_full))
print("partial core:", tuple(core_partial.shape), "factors:", sorted(factors_partial))
for mode, U in factors_full.items():
    print(f"mode={mode}: U.shape={tuple(U.shape)}")

`ranks` に全modeを含めればtruncated HOSVD、一部のmodeだけを含めればpartial HOSVDとして使う。

```python
# 全mode
full_ranks = {0: 3, 1: 4, 2: 2}

# mode 0, 1だけ
partial_ranks = {0: 2, 1: 2}
```

factor matrixもmodeとの対応が失われないよう、`{mode: factor_matrix}` の形で扱う。


### ライブラリでは

PyTorchにはHOSVDそのものを行う関数はなく、行列SVDの `torch.linalg.svd` が自作時の基本部品になる。

```python
U, S, Vh = torch.linalg.svd(matrix, full_matrices=False)
```

TensorLyには全modeのTucker分解を行う `tensorly.decomposition.tucker` と、指定modeだけを分解する `partial_tucker` がある。

```python
from tensorly.decomposition import tucker, partial_tucker

# 全mode
core_tl, factors_tl = tucker(
    X,
    rank=[2, 2, 2],
    init="svd",
)

# mode 0, 1だけ
modes = [0, 1]
core_partial, factors_partial = partial_tucker(
    X,
    rank=[2, 2],
    modes=modes,
    init="svd",
)

# TensorLyはfactorをlistで返すため、mode対応を明示したい場合
factors_partial_by_mode = dict(zip(modes, factors_partial))
```

**注意:** TensorLyの `tucker` / `partial_tucker` はHigher Order Orthogonal Iteration (HOI/HOOI)によるTucker分解で、自作する一回のtruncated HOSVDと完全に同じ処理ではない。比較ではfactorの値の完全一致ではなく、shape・再構成結果・再構成誤差を確認する。


## 6. Tucker再構成


In [7]:
# 実装は nn_compression/compression/tucker.py の reconstruct_tucker
print("full   :", tuple(reconstruct_tucker(core_full, factors_full).shape))
print("partial:", tuple(reconstruct_tucker(core_partial, factors_partial).shape))

### ライブラリでは

全modeの通常のTucker分解なら、TensorLyの `tensorly.tucker_tensor.tucker_to_tensor` を使える。

```python
from tensorly.tucker_tensor import tucker_to_tensor

X_hat_tl = tucker_to_tensor((core_tl, factors_tl))
```

部分modeだけを分解した `partial_tucker` の結果は、指定したmodeにfactorを掛け戻せば再構成できる。

```python
from tensorly.tenalg import multi_mode_dot

X_hat_partial = multi_mode_dot(
    core_partial,
    factors_partial,
    modes=modes,
)
```

PyTorchにはTucker形式の `core + factors` を直接受け取って再構成する専用APIはない。


## 7. 再構成誤差


In [8]:
# 実装は nn_compression/metrics/tensor_approximation.py の relative_frobenius_error
print("full rank :", relative_frobenius_error(X, reconstruct_tucker(*hosvd(X, {0: 3, 1: 4, 2: 2}))).item())
print("low rank  :", relative_frobenius_error(X, reconstruct_tucker(*hosvd(X, {0: 2, 1: 1, 2: 2}))).item())

完全rankで全modeをHOSVDした場合は、数値誤差を除いて再構成誤差が十分小さくなることを確認する。

\[
\frac{\|X-\hat X\|_F}{\|X\|_F}
\]

PyTorchではFrobenius normの計算に `torch.linalg.vector_norm` などを利用できる。


## 8. factor matrixの直交性


In [9]:
# これは学習時の確認用なので src へは移さず、このNotebookに残す。
def orthogonality_error(U: torch.Tensor) -> torch.Tensor:
    """
    factor matrix U の列がどの程度直交規格化されているかを返す。

    完全に直交規格化されていれば 0 に近い値になる。
    """
    if U.ndim != 2:
        raise ValueError("Uは2次元行列で指定してください。")

    identity = torch.eye(
        U.shape[1],
        dtype=U.dtype,
        device=U.device,
    )

    return torch.linalg.matrix_norm(U.T @ U - identity, ord="fro")


HOSVDで得たfactor matrixについて、

\[
U^\mathsf{T}U \simeq I
\]

となることを確認する。

ここでも行列積やnorm自体はPyTorchの基本演算を使ってよい。


## 9. 自作実装の確認

次の順で確認する。

1. 各modeで `fold(unfold(X, mode), mode, X.shape)` が `X` に戻る
2. 全modeを完全rankでHOSVDしたとき再構成誤差が十分小さい
3. rankを下げると再構成誤差が変化する
4. 一部modeだけを指定したpartial HOSVDでも元shapeへ再構成できる
5. `factors` のkeyと元テンソルのmodeの対応が保たれている
6. 各factor matrixの直交性を確認する


In [22]:
print(f"Ans1:{fold(unfold(X,1),1,X.shape)}")
"------------------------------"
rank:dict[int,int]={
    0:3,
    1:4,
    2:2
}
print(f"Ans2:{relative_frobenius_error(X,reconstruct_tucker(*hosvd(X,rank)))}")
"------------------------------"
rank:dict[int,int]={
    0:2,
    1:1,
    2:2
}
print(f"Ans3:{relative_frobenius_error(X,reconstruct_tucker(*hosvd(X,rank)))}")
"------------------------------"
rank:dict[int,int]={
    0:3,
    2:2
}
print(f"Ans4:{reconstruct_tucker(*hosvd(X,rank))}")
"------------------------------"
_,factors=hosvd(X,rank)
for mode,U in factors.items():
    print(f"Ans6:mode->{mode},orthogonality_error->{orthogonality_error(U)}")

Ans1:tensor([[[1., 2.],
         [3., 4.],
         [5., 6.],
         [7., 8.]],

        [[2., 1.],
         [4., 3.],
         [6., 5.],
         [8., 7.]],

        [[1., 3.],
         [2., 4.],
         [3., 5.],
         [4., 6.]]])
Ans2:2.2447464687047614e-07
Ans3:0.0880948156118393
Ans4:tensor([[[1.0000, 2.0000],
         [3.0000, 4.0000],
         [5.0000, 6.0000],
         [7.0000, 8.0000]],

        [[2.0000, 1.0000],
         [4.0000, 3.0000],
         [6.0000, 5.0000],
         [8.0000, 7.0000]],

        [[1.0000, 3.0000],
         [2.0000, 4.0000],
         [3.0000, 5.0000],
         [4.0000, 6.0000]]])
Ans6:mode->0,orthogonality_error->1.884864389012364e-07
Ans6:mode->2,orthogonality_error->3.3717478231665154e-07


## 10. TensorLyとの照合

自作実装が完成したら、同じテンソルをTensorLyでも処理し、次を比較する。

- unfolding / foldingのshapeと値
- full / partialでのcore tensorのshape
- factor matrixのshapeとmode対応
- 再構成テンソル
- 再構成誤差
- factor matrixの直交性

factor matrixそのものは符号や基底の取り方が異なる場合があるため、要素ごとの完全一致だけを正解条件にはしない。


In [39]:
import tensorly as tl
from tensorly.decomposition import tucker, partial_tucker
from tensorly.tucker_tensor import tucker_to_tensor
from tensorly.tenalg import multi_mode_dot

tl.set_backend("pytorch")

# --- unfolding / folding の shape と値 ---
for mode in range(X.ndim):
    X_unf = unfold(X, mode)
    X_unf_tl = tl.unfold(X, mode)
    X_fold = fold(X_unf, mode, X.shape)
    X_fold_tl = tl.fold(X_unf_tl, mode, X.shape)
    print(
        f"mode={mode}: unfold close={torch.allclose(X_unf, X_unf_tl)}, "
        f"fold close={torch.allclose(X_fold, X_fold_tl)}, "
        f"unfold shape={tuple(X_unf.shape)}"
    )

# --- full: core / factors の shape、再構成、誤差、直交性 ---
full_ranks = {0: 2, 1: 2, 2: 2}
core, factors = hosvd(X, full_ranks)
X_hat = reconstruct_tucker(core, factors)

core_tl, factors_tl = tucker(X, rank=[2, 2, 2], init="svd")
X_hat_tl = tucker_to_tensor((core_tl, factors_tl))

print("full core shape: self=", tuple(core.shape), "tl=", tuple(core_tl.shape))
for mode, U in factors.items():
    print(
        f"full factor mode={mode}: self={tuple(U.shape)}, "
        f"tl={tuple(factors_tl[mode].shape)}, "
        f"orth self={orthogonality_error(U).item():.3e}, "
        f"tl={orthogonality_error(factors_tl[mode]).item():.3e}"
    )
print(
    "full recon error: self=",
    relative_frobenius_error(X, X_hat).item(),
    "tl=",
    relative_frobenius_error(X, X_hat_tl).item(),
)

# --- partial: core shape、mode対応、再構成 ---
modes = [0, 1]
partial_ranks = {0: 2, 1: 2}
core_p, factors_p = hosvd(X, partial_ranks)
X_hat_p = reconstruct_tucker(core_p, factors_p)

(core_partial, factors_partial), _ = partial_tucker(
    X,
    rank=[2, 2],
    modes=modes,
    init="svd",
)
factors_partial_by_mode = dict(zip(modes, factors_partial))
X_hat_partial = multi_mode_dot(core_partial, factors_partial, modes=modes)

print("partial core shape: self=", tuple(core_p.shape), "tl=", tuple(core_partial.shape))
print("partial factor keys: self=", sorted(factors_p), "tl=", sorted(factors_partial_by_mode))
for mode in modes:
    print(
        f"partial factor mode={mode}: self={tuple(factors_p[mode].shape)}, "
        f"tl={tuple(factors_partial_by_mode[mode].shape)}"
    )
print(
    "partial recon error: self=",
    relative_frobenius_error(X, X_hat_p).item(),
    "tl=",
    relative_frobenius_error(X, X_hat_partial).item(),
)

mode=0: unfold close=True, fold close=True, unfold shape=(3, 8)
mode=1: unfold close=True, fold close=True, unfold shape=(4, 6)
mode=2: unfold close=True, fold close=True, unfold shape=(2, 12)
full core shape: self= (2, 2, 2) tl= (2, 2, 2)
full factor mode=0: self=(3, 2), tl=(3, 2), orth self=1.738e-07, tl=2.384e-07
full factor mode=1: self=(4, 2), tl=(4, 2), orth self=3.863e-07, tl=4.787e-07
full factor mode=2: self=(2, 2), tl=(2, 2), orth self=3.372e-07, tl=1.686e-07
full recon error: self= 0.03588838502764702 tl= 0.03588837385177612
partial core shape: self= (2, 2, 2) tl= (2, 2, 2)
partial factor keys: self= [0, 1] tl= [0, 1]
partial factor mode=0: self=(3, 2), tl=(3, 2)
partial factor mode=1: self=(4, 2), tl=(4, 2)
partial recon error: self= 0.03588838502764702 tl= 0.03588838502764702
